In [ ]:
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install('pyb2d-jupyterlite-backend==0.4.2')

# Pyodide's b2d package references an optional module that is absent from its
# wheel. Register only the missing symbol before loading the JupyterLite backend.
import types
from pathlib import Path
import b2d

_compat_name = "b2d.testbed.backend.jupyter.async_jupyter_gui"
_compat_path = (
    Path(b2d.__file__).parent
    / "testbed/backend/jupyter/async_jupyter_gui.py"
)
if not _compat_path.exists() and _compat_name not in sys.modules:
    _compat_module = types.ModuleType(_compat_name)
    _compat_module.JupyterAsyncGui = object
    sys.modules[_compat_name] = _compat_module

# The legacy renderer is not safe inside modern Pyodide workers. Keep
# physics code runnable while suppressing unsupported debug callbacks.
import pyb2d_jupyterlite_backend.plot as _backend_plot
from pyb2d_jupyterlite_backend.async_jupyter_gui import JupyterAsyncGui as _BackendGui
from IPython.display import display

_original_render_world = _backend_plot.render_world
_original_animate_world = _backend_plot.animate_world
_original_gui_init = _BackendGui.__init__

def _render_supported_layers(*args, **kwargs):
    kwargs["flags"] = []
    return _original_render_world(*args, **kwargs)

def _plot_supported_layers(*args, **kwargs):
    display(_render_supported_layers(*args, **kwargs))

def _animate_supported_layers(*args, **kwargs):
    kwargs["flags"] = []
    return _original_animate_world(*args, **kwargs)

def _gui_without_shape_batches(self, *args, **kwargs):
    _original_gui_init(self, *args, **kwargs)
    self._debug_draw_flags = []

_backend_plot.render_world = _render_supported_layers
_backend_plot.plot_world = _plot_supported_layers
_backend_plot.animate_world = _animate_supported_layers
_BackendGui.__init__ = _gui_without_shape_batches
b2d.plot.render_world = _render_supported_layers
b2d.plot.plot_world = _plot_supported_layers
b2d.plot.animate_world = _animate_supported_layers


# Initialize each scene without starting the legacy infinite asyncio loop.
import pyb2d_jupyterlite_backend.async_jupyter_gui as _backend_gui

def _finite_start_ui(self):
    self.canvas = _backend_gui.Canvas(
        width=self.resolution[0], height=self.resolution[1]
    )
    self.out = _backend_gui.ipywidgets.Output()
    self._setup_ipywidgets_gui()
    self.make_testbed()
    self._events = []
    self._stop = True
    return None

_BackendGui.start_ui = _finite_start_ui
print("PyB2D compatibility mode: scene initialized; legacy interactive drawing is disabled.")


In [ ]:
from b2d.testbed import TestbedBase
import random
import numpy
import b2d


class GaussMachine(TestbedBase):

    name = "Gauss Machine"

    def __init__(self, settings=None):
        super(GaussMachine, self).__init__(settings=settings)

        self.box_shape = 30, 20
        box_shape = self.box_shape

        # outer box
        verts = numpy.array(
            [(0, box_shape[1]), (0, 0), (box_shape[0], 0), (box_shape[0], box_shape[1])]
        )
        shape = b2d.chain_shape(vertices=numpy.flip(verts, axis=0))
        box = self.world.create_static_body(position=(0, 0), shape=shape)

        # "bins"
        bin_height = box_shape[1] / 3
        bin_width = 1
        for x in range(0, box_shape[0], bin_width):
            box = self.world.create_static_body(
                position=(0, 0), shape=b2d.two_sided_edge_shape((x, 0), (x, bin_height))
            )

        # reflectors
        ref_start_y = int(bin_height + box_shape[1] / 10.0)
        ref_stop_y = int(box_shape[1] * 0.9)
        for x in range(0, box_shape[0] + 1):

            for y in range(ref_start_y, ref_stop_y):
                s = [0.5, 0][y % 2 == 0]
                shape = b2d.circle_shape(radius=0.3)
                box = self.world.create_static_body(position=(x + s, y), shape=shape)

        # particle system
        pdef = b2d.particle_system_def(
            viscous_strength=0.9,
            spring_strength=0.0,
            damping_strength=100.5,
            pressure_strength=1.0,
            color_mixing_strength=0.05,
            density=2,
        )

        psystem = self.world.create_particle_system(pdef)
        psystem.radius = 0.1
        psystem.damping = 0.5

        # linear emitter
        emitter_pos = (self.box_shape[0] / 2, self.box_shape[1] + 10)
        emitter_def = b2d.RandomizedLinearEmitterDef()
        emitter_def.emite_rate = 400
        emitter_def.lifetime = 25
        emitter_def.size = (10, 1)
        emitter_def.transform = b2d.Transform(emitter_pos, b2d.Rot(0))

        self.emitter = b2d.RandomizedLinearEmitter(psystem, emitter_def)

    def pre_step(self, dt):
        self.emitter.step(dt)

In [ ]:
from pyb2d_jupyterlite_backend.async_jupyter_gui import JupyterAsyncGui
s = JupyterAsyncGui.Settings()
s.resolution = [350,400]
s.scale = 11
tb = b2d.testbed.run(GaussMachine, backend=JupyterAsyncGui, gui_settings=s);